# ✈️ Airport Arrival Optimizer
### Know exactly when to leave for the airport: based on real airline data

**Data sources:**
- On-time performance: Cirium 2024 On-Time Performance Review & U.S. DOT Air Travel Consumer Report
- TSA wait times: TSA historical patterns (MyTSA app data)
- Boarding window estimates: standard airline policies

---

##Install dependencies

In [ ]:
# !pip install pandas numpy matplotlib seaborn

##Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Baby-blue color palette
PALETTE = {
    'bg':        '#EAF4FB',
    'card':      '#FFFFFF',
    'blue1':     '#AED6F1',
    'blue2':     '#5DADE2',
    'blue3':     '#2E86C1',
    'accent':    '#1A5276',
    'green':     '#58D68D',
    'amber':     '#F4D03F',
    'red':       '#EC7063',
    'text':      '#1A252F',
    'subtext':   '#5D6D7E',
}

plt.rcParams.update({
    'figure.facecolor':  PALETTE['bg'],
    'axes.facecolor':    PALETTE['card'],
    'font.family':       'DejaVu Sans',
    'axes.titlesize':    13,
    'axes.labelsize':    11,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

print("Libraries loaded.")

##Airline database
Real 2024 on-time performance data (Cirium / DOT ATCR) + airline-specific parameters.

In [ ]:
AIRLINES = pd.DataFrame([
    # name, code, on_time_pct, avg_delay_min (when late), gate_close_min, board_start_min, tsa_precheck, hub_airports
    ('Delta Air Lines',      'DL', 83.46, 42, 10, 35, True,  'ATL,JFK,LAX,DTW,MSP,SLC'),
    ('United Airlines',      'UA', 80.93, 47, 10, 35, True,  'ORD,EWR,IAH,DEN,SFO,LAX'),
    ('Alaska Airlines',      'AS', 79.25, 38, 10, 30, True,  'SEA,LAX,SFO,PDX,ANC'),
    ('American Airlines',    'AA', 77.78, 51, 10, 40, True,  'DFW,CLT,PHX,MIA,ORD,JFK'),
    ('Southwest Airlines',   'WN', 77.77, 44, 10, 30, False, 'DAL,MDW,BWI,HOU,LAS,DEN'),
    ('JetBlue Airways',      'B6', 74.20, 55, 10, 30, True,  'JFK,BOS,FLL,LGB,MCO'),
    ('Spirit Airlines',      'NK', 72.10, 50, 15, 45, False, 'FLL,LAS,MCO,ATL,DFW'),
    ('Frontier Airlines',    'F9', 71.50, 53, 15, 45, False, 'DEN,MCO,LAS,PHX,CLT'),
    ('Hawaiian Airlines',    'HA', 80.10, 35, 10, 30, True,  'HNL,OGG,KOA,LIH,ITO'),
    ('Allegiant Air',        'G4', 68.90, 58, 20, 50, False, 'LAS,PIE,AZA,SFB,VPS'),
], columns=['airline', 'code', 'on_time_pct', 'avg_delay_when_late',
            'gate_close_min', 'board_start_min', 'precheck_partner', 'hubs'])

print(AIRLINES[['airline','code','on_time_pct','avg_delay_when_late','gate_close_min']].to_string(index=False))

##TSA wait time model
Hour-by-hour wait time patterns based on MyTSA historical data.

In [ ]:
# Average TSA standard lane wait times by hour (minutes), major US hubs
# Source: TSA/MyTSA historical patterns + airport-reported data
TSA_HOURLY = {
    h: w for h, w in enumerate([
        8,   # 0:00 — near empty
        5,   # 1:00
        5,   # 2:00
        5,   # 3:00
        10,  # 4:00 — early birds
        22,  # 5:00 — morning rush builds
        35,  # 6:00 — peak AM
        42,  # 7:00 — peak AM
        38,  # 8:00
        28,  # 9:00 — tapering
        20,  # 10:00
        18,  # 11:00
        22,  # 12:00 — midday slight rise
        24,  # 13:00
        26,  # 14:00
        30,  # 15:00 — afternoon wave
        36,  # 16:00 — peak PM
        40,  # 17:00 — peak PM
        34,  # 18:00
        25,  # 19:00
        18,  # 20:00
        14,  # 21:00
        10,  # 22:00
        8,   # 23:00
    ])
}

TSA_PRECHECK_MULTIPLIER = 0.22   # PreCheck ~78% faster (TSA: <10 min for 95%+ of users)
TSA_CLEAR_MULTIPLIER    = 0.10   # CLEAR biometric virtually instant

def tsa_wait(hour: int, has_precheck: bool = False, has_clear: bool = False) -> int:
    base = TSA_HOURLY.get(hour % 24, 20)
    if has_clear:
        return max(3, int(base * TSA_CLEAR_MULTIPLIER))
    if has_precheck:
        return max(5, int(base * TSA_PRECHECK_MULTIPLIER))
    return base

# Quick test
for h in [5, 7, 12, 17, 22]:
    print(f"  {h:02d}:00 — Standard: {tsa_wait(h):2d} min | PreCheck: {tsa_wait(h, True):2d} min | CLEAR: {tsa_wait(h, has_clear=True):2d} min")

##Core optimizer function

In [ ]:
def optimize_airport_arrival(
    flight_time: str,          # 'HH:MM'
    airline_code: str,         # e.g. 'DL'
    flight_type: str = 'domestic',   # 'domestic' or 'international'
    has_checked_bag: bool = False,
    has_precheck: bool = False,
    has_clear: bool = False,
    day_of_week: str = 'Monday',     # affects TSA crowding
    is_peak_season: bool = False,
    parking_walk_min: int = 10,
) -> dict:
    """
    Returns recommended airport arrival time + breakdown of time components.
    """
    airline = AIRLINES[AIRLINES['code'] == airline_code].iloc[0]

    ft = datetime.strptime(flight_time, '%H:%M')
    hour = ft.hour

    # --- TSA wait ---
    tsa_min = tsa_wait(hour, has_precheck, has_clear)

    # Peak season / weekend adjustment
    if is_peak_season:
        tsa_min = int(tsa_min * 1.35)
    if day_of_week in ['Friday', 'Sunday']:
        tsa_min = int(tsa_min * 1.20)
    elif day_of_week == 'Monday':
        tsa_min = int(tsa_min * 1.10)

    # --- Bag drop ---
    bag_drop_min = 20 if has_checked_bag else 0

    # --- Gate buffer (close time + walk) ---
    gate_close_min   = airline['gate_close_min']
    board_start_min  = airline['board_start_min']
    gate_walk_min    = 10  # avg terminal walk

    # --- International adds customs/immigration buffer ---
    intl_buffer_min = 45 if flight_type == 'international' else 0

    # --- Delay risk buffer ---
    on_time = airline['on_time_pct'] / 100
    delay_risk_buffer = int((1 - on_time) * airline['avg_delay_when_late'] * 0.5)
    # (we don't pad arrival for airline delays — these affect when you LEAVE, not when you arrive)

    total_buffer_min = (
        parking_walk_min
        + bag_drop_min
        + tsa_min
        + gate_walk_min
        + board_start_min
        + intl_buffer_min
        + 5  # check-in kiosk / misc
    )

    recommended_arrival = ft - timedelta(minutes=total_buffer_min)
    latest_arrival      = ft - timedelta(minutes=tsa_min + gate_close_min + gate_walk_min + bag_drop_min + 5)

    return {
        'airline': airline['airline'],
        'flight_time': flight_time,
        'recommended_arrival': recommended_arrival.strftime('%H:%M'),
        'latest_safe_arrival': latest_arrival.strftime('%H:%M'),
        'total_buffer_min': total_buffer_min,
        'breakdown': {
            'Parking / walk-in':   parking_walk_min,
            'Bag drop':            bag_drop_min,
            'TSA security':        tsa_min,
            'Terminal walk':       gate_walk_min,
            'Board start buffer':  board_start_min,
            'International extra': intl_buffer_min,
            'Misc check-in':       5,
        },
        'on_time_pct': airline['on_time_pct'],
        'delay_risk_pct': round(100 - airline['on_time_pct'], 2),
        'precheck': has_precheck,
        'clear': has_clear,
    }

# --- Quick demo ---
result = optimize_airport_arrival(
    flight_time='08:00',
    airline_code='DL',
    has_precheck=True,
    day_of_week='Friday',
)
print(f"Airline          : {result['airline']}")
print(f"Flight departs   : {result['flight_time']}")
print(f"Recommended arr. : {result['recommended_arrival']}")
print(f"Latest safe arr. : {result['latest_safe_arrival']}")
print(f"Total buffer     : {result['total_buffer_min']} min")
print(f"On-time record   : {result['on_time_pct']}%")
print("\nBreakdown:")
for k, v in result['breakdown'].items():
    print(f"  {k:25s}: {v} min")

##Compare all 10 airlines for a given flight

In [ ]:
def compare_all_airlines(
    flight_time: str = '08:00',
    flight_type: str = 'domestic',
    has_checked_bag: bool = True,
    has_precheck: bool = False,
    day_of_week: str = 'Monday',
    is_peak_season: bool = False,
) -> pd.DataFrame:
    rows = []
    for _, airline in AIRLINES.iterrows():
        r = optimize_airport_arrival(
            flight_time=flight_time,
            airline_code=airline['code'],
            flight_type=flight_type,
            has_checked_bag=has_checked_bag,
            has_precheck=has_precheck,
            day_of_week=day_of_week,
            is_peak_season=is_peak_season,
        )
        rows.append({
            'Airline':            r['airline'],
            'Code':               airline['code'],
            'Recommended Arrive': r['recommended_arrival'],
            'Latest Safe Arrive': r['latest_safe_arrival'],
            'Buffer (min)':       r['total_buffer_min'],
            'On-Time %':          airline['on_time_pct'],
            'Delay Risk %':       r['delay_risk_pct'],
            'PreCheck Partner':   '✓' if airline['precheck_partner'] else '✗',
        })
    return pd.DataFrame(rows).sort_values('On-Time %', ascending=False).reset_index(drop=True)

df = compare_all_airlines(flight_time='08:00', has_checked_bag=True, day_of_week='Friday', is_peak_season=True)
print(df.to_string(index=False))

##Visualizations

In [ ]:
#Figure 1: On-time performance ranking ──
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(PALETTE['bg'])
ax.set_facecolor(PALETTE['card'])

colors = [PALETTE['blue3'] if v >= 80 else PALETTE['blue2'] if v >= 75 else PALETTE['blue1']
          for v in df['On-Time %']]

bars = ax.barh(df['Airline'], df['On-Time %'], color=colors, height=0.6, zorder=2)
ax.axvline(78, color=PALETTE['accent'], linestyle='--', linewidth=1.2, alpha=0.6, label='Industry avg 78%')

for bar, val in zip(bars, df['On-Time %']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9, color=PALETTE['accent'], fontweight='bold')

ax.set_xlabel('On-Time Arrival Rate (%)', color=PALETTE['text'])
ax.set_title('2024 On-Time Performance — Top 10 U.S. Airlines', color=PALETTE['accent'],
             fontweight='bold', pad=12)
ax.set_xlim(60, 90)
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3, zorder=0)
ax.tick_params(colors=PALETTE['text'])
plt.tight_layout()
plt.show()

In [ ]:
#Figure 2: TSA wait times by hour (standard vs PreCheck) ──
hours  = list(range(24))
std    = [TSA_HOURLY[h] for h in hours]
pre    = [tsa_wait(h, has_precheck=True) for h in hours]
labels = [f'{h:02d}:00' for h in hours]

fig, ax = plt.subplots(figsize=(12, 4))
fig.patch.set_facecolor(PALETTE['bg'])
ax.set_facecolor(PALETTE['card'])

ax.fill_between(hours, std, alpha=0.25, color=PALETTE['blue3'])
ax.plot(hours, std, color=PALETTE['blue3'], linewidth=2.2, label='Standard lane')
ax.fill_between(hours, pre, alpha=0.25, color=PALETTE['green'])
ax.plot(hours, pre, color=PALETTE['green'], linewidth=2.2, label='TSA PreCheck', linestyle='--')

# Shade peak zones
ax.axvspan(5, 8,  alpha=0.08, color=PALETTE['red'],   label='AM peak')
ax.axvspan(15, 18, alpha=0.08, color=PALETTE['amber'], label='PM peak')

ax.set_xticks(hours[::2])
ax.set_xticklabels(labels[::2], rotation=45, ha='right')
ax.set_ylabel('Est. wait time (minutes)')
ax.set_title('TSA Security Wait Times by Hour — Major U.S. Airports', color=PALETTE['accent'],
             fontweight='bold', pad=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.tick_params(colors=PALETTE['text'])
plt.tight_layout()
plt.show()

In [ ]:
#Figure 3: Buffer breakdown stacked bar for all airlines ──
FLIGHT_TIME = '07:30'
HAS_BAG     = True
PRECHECK    = False
DOW         = 'Friday'
PEAK        = True

results = []
for _, airline in AIRLINES.iterrows():
    r = optimize_airport_arrival(
        FLIGHT_TIME, airline['code'],
        has_checked_bag=HAS_BAG,
        has_precheck=PRECHECK,
        day_of_week=DOW,
        is_peak_season=PEAK,
    )
    row = {'Airline': airline['code']}
    row.update(r['breakdown'])
    results.append(row)

stacked = pd.DataFrame(results).set_index('Airline')
stacked = stacked.loc[df['Code'].values]  # sort by on-time

component_colors = ['#AED6F1','#5DADE2','#2E86C1','#1A5276','#85C1E9','#D6EAF8','#0E6655']

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor(PALETTE['bg'])
ax.set_facecolor(PALETTE['card'])

stacked.plot(kind='bar', stacked=True, ax=ax, color=component_colors, width=0.65, zorder=2)

ax.set_title(f'Airport Arrival Buffer Breakdown — {FLIGHT_TIME} Flight, {DOW}, Peak Season={PEAK}',
             color=PALETTE['accent'], fontweight='bold', pad=12)
ax.set_xlabel('Airline Code')
ax.set_ylabel('Minutes before departure')
ax.legend(loc='upper right', fontsize=8, framealpha=0.8)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.tick_params(axis='x', rotation=0, colors=PALETTE['text'])
plt.tight_layout()
plt.show()

In [ ]:
#Figure 4: Delay risk heatmap by airline × hour of day ──
hours_day = list(range(5, 23))
heat_data = []

for _, airline in AIRLINES.sort_values('on_time_pct', ascending=False).iterrows():
    row = []
    for h in hours_day:
        r = optimize_airport_arrival(
            f'{h:02d}:00', airline['code'],
            has_precheck=False, has_checked_bag=False, day_of_week='Monday'
        )
        row.append(r['total_buffer_min'])
    heat_data.append(row)

heat_df = pd.DataFrame(
    heat_data,
    index=AIRLINES.sort_values('on_time_pct', ascending=False)['code'],
    columns=[f'{h:02d}:00' for h in hours_day]
)

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor(PALETTE['bg'])

cmap = sns.color_palette("Blues", as_cmap=True)
sns.heatmap(heat_df, ax=ax, cmap=cmap, annot=True, fmt='d',
            linewidths=0.4, linecolor='#D6EAF8',
            cbar_kws={'label': 'Recommended buffer (min)'})

ax.set_title('Recommended Arrival Buffer (min) by Airline & Departure Hour — No Bag, No PreCheck',
             color=PALETTE['accent'], fontweight='bold', pad=12)
ax.set_xlabel('Departure hour')
ax.set_ylabel('Airline')
plt.tight_layout()
plt.show()

## 7. Personal trip planner — run this for your flight

In [ ]:
# ============================================================
# ✏️  FILL IN YOUR TRIP DETAILS BELOW
# ============================================================

MY_FLIGHT_TIME   = '09:30'          # HH:MM (24hr)
MY_AIRLINE       = 'AA'             # DL UA AS AA WN B6 NK F9 HA G4
MY_FLIGHT_TYPE   = 'domestic'       # 'domestic' or 'international'
MY_CHECKED_BAG   = True             # True / False
MY_PRECHECK      = False            # True / False
MY_CLEAR         = False            # True / False
MY_DAY           = 'Monday'         # Monday Tuesday Wednesday Thursday Friday Saturday Sunday
MY_PEAK_SEASON   = False            # True during summer, holidays, spring break
MY_PARKING_WALK  = 12               # minutes from car/rideshare to check-in

# ============================================================

res = optimize_airport_arrival(
    flight_time=MY_FLIGHT_TIME,
    airline_code=MY_AIRLINE,
    flight_type=MY_FLIGHT_TYPE,
    has_checked_bag=MY_CHECKED_BAG,
    has_precheck=MY_PRECHECK,
    has_clear=MY_CLEAR,
    day_of_week=MY_DAY,
    is_peak_season=MY_PEAK_SEASON,
    parking_walk_min=MY_PARKING_WALK,
)

print("=" * 50)
print(f"  ✈  {res['airline']}  |  Flight at {res['flight_time']}")
print("=" * 50)
print(f"  Recommended arrival : {res['recommended_arrival']}")
print(f"  Latest safe arrival : {res['latest_safe_arrival']}")
print(f"  Total buffer needed : {res['total_buffer_min']} minutes")
print(f"  On-time rate        : {res['on_time_pct']}%")
print(f"  Delay risk          : {res['delay_risk_pct']}%")
print()
print("  Time breakdown:")
for component, mins in res['breakdown'].items():
    if mins > 0:
        bar = '█' * (mins // 2)
        print(f"    {component:25s}  {mins:3d} min  {bar}")

# Pie chart of breakdown
breakdown = {k: v for k, v in res['breakdown'].items() if v > 0}
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor(PALETTE['bg'])
ax.set_facecolor(PALETTE['bg'])

wedge_colors = ['#AED6F1','#5DADE2','#2E86C1','#1A5276','#85C1E9','#D6EAF8']
wedges, texts, autotexts = ax.pie(
    breakdown.values(),
    labels=breakdown.keys(),
    colors=wedge_colors[:len(breakdown)],
    autopct='%1.0f%%',
    startangle=140,
    pctdistance=0.8,
)
for t in autotexts:
    t.set_fontsize(9)
    t.set_color('white')

ax.set_title(f'Your Arrival Buffer — {res["airline"]} @ {res["flight_time"]}',
             color=PALETTE['accent'], fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Best time of day to fly — TSA crowd score by hour

In [ ]:
hours  = list(range(24))
std    = [TSA_HOURLY[h] for h in hours]

def crowd_label(w):
    if w <= 12: return 'Best'
    if w <= 25: return 'Good'
    if w <= 35: return 'Fair'
    return 'Avoid'

crowd_colors = {
    'Best':  PALETTE['green'],
    'Good':  PALETTE['blue2'],
    'Fair':  PALETTE['amber'],
    'Avoid': PALETTE['red'],
}

fig, ax = plt.subplots(figsize=(12, 3.5))
fig.patch.set_facecolor(PALETTE['bg'])
ax.set_facecolor(PALETTE['card'])

for h, w in zip(hours, std):
    label = crowd_label(w)
    ax.bar(h, w, color=crowd_colors[label], width=0.85, zorder=2)

ax.set_xticks(hours)
ax.set_xticklabels([f'{h:02d}' for h in hours], fontsize=8)
ax.set_ylabel('TSA wait (min)')
ax.set_title('Best Hours to Clear TSA Security — Standard Lane', color=PALETTE['accent'], fontweight='bold')

legend_patches = [mpatches.Patch(color=c, label=l) for l, c in crowd_colors.items()]
ax.legend(handles=legend_patches, loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.3, zorder=0)
plt.tight_layout()
plt.show()

print("\nRecommended departure times (TSA wait under 15 min):")
good_hours = [f'{h:02d}:00' for h, w in zip(hours, std) if w <= 15]
print('  ' + '  '.join(good_hours))

---
## Airline quick reference

| Code | Airline | On-Time 2024 | Gate Closes | PreCheck |
|------|---------|-------------|-------------|----------|
| DL | Delta Air Lines | 83.5% | 10 min | ✓ |
| UA | United Airlines | 80.9% | 10 min | ✓ |
| HA | Hawaiian Airlines | 80.1% | 10 min | ✓ |
| AS | Alaska Airlines | 79.3% | 10 min | ✓ |
| AA | American Airlines | 77.8% | 10 min | ✓ |
| WN | Southwest Airlines | 77.8% | 10 min | ✗ |
| B6 | JetBlue Airways | 74.2% | 10 min | ✓ |
| NK | Spirit Airlines | 72.1% | 15 min | ✗ |
| F9 | Frontier Airlines | 71.5% | 15 min | ✗ |
| G4 | Allegiant Air | 68.9% | 20 min | ✗ |

*On-time = arrival within 15 min of scheduled gate time. Source: Cirium 2024 OTP Review.*